# Customer Response to Marketing Campaign

**Tabular Classification Project**

## 2 · Project Overview

Predict whether a customer will **respond positively** to a marketing campaign based on demographics, purchase history, and campaign interaction features. The dataset has ~6,000 customers with ~14% response rate — a realistic marketing analytics classification problem.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given customer demographics (age, income, education), purchase behavior (recency, frequency, monetary), and campaign details (channel, offer type, previous contacts), predict whether they will respond (responded = 1).

## 5 · Why This Project Matters

- Marketing campaign optimization can **save 30-50%** of campaign budgets.
- Response rate prediction enables targeted outreach and better ROI.
- ~14% response rate is typical for direct marketing campaigns.
- This teaches **RFM analysis** (Recency, Frequency, Monetary) — a marketing ML staple.
- Business context (cost per contact, lifetime value) drives model decisions.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~6,000 |
| Features | 13 (age, income, education, marital_status, recency, frequency, monetary, campaign_channel, offer_type, num_previous_contacts, days_since_last_contact, num_children, web_visits_monthly) |
| Target | `responded` (binary: 0=no response, 1=responded) |
| Class balance | ~86% no response, ~14% responded |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: Synthetic marketing campaign response dataset for ML practice.
**License**: Educational / public.
**Note**: Inspired by direct marketing datasets (KDD Cup, Kaggle marketing data).

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "responded"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Customer Response to Marketing Campaign


## 11 · Dataset Download or Loading

In [4]:
np.random.seed(SEED)
n = 6000

age = np.random.normal(48, 14, n).clip(18, 85).astype(int)
income = np.random.lognormal(10.7, 0.6, n).clip(15000, 200000).round(0).astype(int)
education = np.random.choice(['Basic', 'High School', 'Graduation', 'Master', 'PhD'], n,
                              p=[0.05, 0.2, 0.4, 0.25, 0.1])
marital = np.random.choice(['Single', 'Married', 'Divorced', 'Widowed'], n,
                            p=[0.25, 0.5, 0.15, 0.1])
recency = np.random.exponential(30, n).clip(0, 100).astype(int)
frequency = np.random.poisson(8, n).clip(0, 30)
monetary = np.random.lognormal(5.5, 1.0, n).clip(10, 5000).round(2)
channel = np.random.choice(['email', 'phone', 'direct_mail', 'social_media'], n,
                            p=[0.35, 0.25, 0.2, 0.2])
offer = np.random.choice(['discount', 'bogo', 'free_trial', 'loyalty_points'], n,
                          p=[0.3, 0.25, 0.2, 0.25])
prev_contacts = np.random.poisson(2, n).clip(0, 10)
days_since_last = np.random.exponential(50, n).clip(1, 365).astype(int)
children = np.random.choice([0, 1, 2, 3], n, p=[0.3, 0.35, 0.25, 0.1])
web_visits = np.random.poisson(5, n).clip(0, 20)

score = (
    0.00002 * (income - 50000)
    + 0.1 * (education == 'PhD').astype(float)
    + 0.05 * (education == 'Master').astype(float)
    - 0.01 * recency
    + 0.03 * frequency
    + 0.0005 * monetary
    + 0.15 * (channel == 'email').astype(float)
    + 0.2 * (offer == 'discount').astype(float)
    + 0.03 * web_visits
    - 0.05 * prev_contacts
    - 0.003 * days_since_last
    + np.random.normal(0, 0.6, n)
)
responded = (score > np.percentile(score, 86)).astype(int)

df = pd.DataFrame({
    'age': age, 'income': income, 'education': education, 'marital_status': marital,
    'recency': recency, 'frequency': frequency, 'monetary': monetary,
    'campaign_channel': channel, 'offer_type': offer,
    'num_previous_contacts': prev_contacts, 'days_since_last_contact': days_since_last,
    'num_children': children, 'web_visits_monthly': web_visits, 'responded': responded,
})
print(f"Dataset shape: {df.shape}")
print(f"Response rate: {df['responded'].mean():.2%}")
df.head()

Dataset shape: (6000, 14)
Response rate: 14.00%


,age,income,education,marital_status,recency,frequency,monetary,campaign_channel,offer_type,num_previous_contacts,days_since_last_contact,num_children,web_visits_monthly,responded
0,54,22733,Graduation,Married,84,7,165.20,social_media,free_trial,4,1,1,9,0
1,46,30377,Graduation,Single,15,8,89.75,email,free_trial,1,12,3,4,0
2,57,25204,Graduation,Married,20,11,79.52,social_media,free_trial,5,257,0,5,0
3,69,31927,Basic,Married,8,6,501.20,phone,free_trial,0,1,2,6,0
4,44,39008,PhD,Divorced,0,13,160.52,direct_mail,loyalty_points,0,3,0,4,0


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (6000, 14)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
responded
0    5160
1     840
Name: count, dtype: int64

Dtypes:
age                          int64
income                       int64
education                   object
marital_status              object
recency                      int64
frequency                    int32
monetary                   float64
campaign_channel            object
offer_type                  object
num_previous_contacts        int32
days_since_last_contact      int64
num_children                 int64
web_visits_monthly           int32
responded                    int64
dtype: object

Target 'responded' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, col in enumerate(['income', 'recency', 'frequency', 'monetary', 'web_visits_monthly', 'age']):
    ax = axes[i // 3, i % 3]
    df[df[TARGET]==0][col].hist(bins=25, ax=ax, alpha=0.6, label='No Response', color='steelblue')
    df[df[TARGET]==1][col].hist(bins=25, ax=ax, alpha=0.6, label='Responded', color='coral')
    ax.set_title(col); ax.legend()
plt.suptitle("Feature Distributions by Response", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(['campaign_channel', 'offer_type', 'education']):
    ct = pd.crosstab(df[col], df[TARGET], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['steelblue', 'coral'])
    axes[i].set_title(f"Response Rate by {col}")
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")

Numeric: 9, Categorical: 4


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title("Campaign Response Distribution")
axes[0].set_xticklabels(['No Response (0)', 'Responded (1)'], rotation=0)
df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts()}")
print(f"\nImbalance ratio: {df[TARGET].value_counts().iloc[0] / max(df[TARGET].value_counts().iloc[1], 1):.0f}:1")

Class distribution:
responded
0    5160
1     840
Name: count, dtype: int64

Imbalance ratio: 6:1


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

Train: (4800, 13), Test: (1200, 13)
Train target dist: {0: 4128, 1: 672}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (13): ['age', 'income', 'education', 'marital_status', 'recency', 'frequency', 'monetary', 'campaign_channel', 'offer_type', 'num_previous_contacts', 'days_since_last_contact', 'num_children', 'web_visits_monthly']


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
y_prob_bl = baseline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.9058
Precision: 0.8977
Recall   : 0.9058
F1       : 0.8982
ROC-AUC  : 0.9035


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
NearestCentroid                0.878333           0.827104  0.916978  0.886120   0.900212  0.878333    0.018395
LinearDiscriminantAnalysis     0.922500           0.773048  0.920606  0.916213   0.917790  0.922500    0.021158
CalibratedClassifierCV         0.921667           0.772564  0.923161  0.915458   0.916740  0.921667    0.051988
LogisticRegression             0.919167           0.768618  0.922873  0.912909   0.913716  0.919167    0.020190
LGBMClassifier                 0.912500           0.762251  0.912231  0.906668   0.905965  0.912500    3.169983
CatBoostClassifier             0.918333           0.760659  0.916223  0.911245   0.912886  0.918333    3.463356
QuadraticDiscriminantAnalysis  0.900833           0.760451  0.898201  

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
_flaml_metric = "macro_f1" if y_train.nunique() > 2 else "f1"
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric=_flaml_metric,
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: lgbm
Accuracy : 0.9125
F1       : 0.9079


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.9133  F1: 0.9061


LightGBM  Acc: 0.9158  F1: 0.9099


XGBoost   Acc: 0.9083  F1: 0.9024


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
LightGBM               0.9158  0.9099     0.9098  0.9158
FLAML                  0.9125  0.9079     0.9065  0.9125
CatBoost               0.9133  0.9061     0.9067  0.9133
XGBoost                0.9083  0.9024     0.9012  0.9083
Logistic Regression    0.9058  0.8982     0.8977  0.9058

Top 3: ['LightGBM', 'FLAML', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  LightGBM
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      1032
           1       0.78      0.56      0.65       168

    accuracy                           0.92      1200
   macro avg       0.85      0.77      0.80      1200
weighted avg       0.91      0.92      0.91      1200


  FLAML
              precision    recall  f1-score   support

           0       0.93      0.97      0.95      1032
           1       0.74      0.58      0.65       168

    accuracy                           0.91      1200
   macro avg       0.84      0.77      0.80      1200
weighted avg       0.91      0.91      0.91      1200


  CatBoost
              precision    recall  f1-score   support

           0       0.93      0.98      0.95      1032
           1       0.78      0.53      0.63       168

    accuracy                           0.91      1200
   macro avg       0.85      0.75      0.79      1200

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: LightGBM
Error rate: 0.0842 (101 / 1200)

Errors by true class:
      errors  total  error_rate
true                           
0         27   1032    0.026163
1         74    168    0.440476


## 25 · Interpretation and Business Insight

- **Recency** (days since last purchase) is a top predictor — recent buyers respond more.
- **Income** and **monetary value** predict willingness to engage.
- **Discount offers** have the highest response rate.
- **Email campaigns** outperform other channels.
- **Web visits** indicate engagement and correlate with response.
- **Previous contacts** have diminishing returns — over-contacting reduces response.

## 26 · Limitations

1. Synthetic data — real marketing depends on timing, personalization, competitive offers.
2. No A/B test structure — can't distinguish selection bias from treatment effect.
3. Missing customer lifecycle stage and product affinity data.
4. No channel interaction effects (email + social combo).
5. Static features — no behavioral sequence data.

## 27 · How to Improve This Project

1. Use uplift modeling to find persuadable customers (not just likely responders).
2. Engineer RFM segments and customer lifetime value.
3. Add channel interaction features.
4. Implement cost-aware classification (cost per contact vs expected revenue).
5. Run A/B tests to measure true causal effect of campaigns.

## 28 · Production Considerations

- Score customers before each campaign for targeting.
- Integrate with CRM and marketing automation platforms.
- A/B test model-targeted vs random campaigns.
- Monitor response rate drift and retrain quarterly.
- Calculate ROI: (revenue from responders - campaign cost) / campaign cost.

## 29 · Common Mistakes

1. Optimizing for response rate instead of profit/ROI.
2. Not considering that some customers would buy anyway (uplift).
3. Over-contacting customers (previous contacts show diminishing returns).
4. Using accuracy on 86/14 split.
5. Not segmenting by customer value (high-value vs low-value responders).

## 30 · Mini Challenge / Exercises

1. Calculate campaign ROI at different targeting thresholds (top 10%, 20%, 50%).
2. Remove income and see if the model still works (privacy constraint).
3. Build an RFM score and use it as a single feature.
4. Compare email-only vs all-channel model performance.

## 31 · Final Summary / Key Takeaways

1. Marketing response prediction is a core business analytics application.
2. RFM features (Recency, Frequency, Monetary) are the strongest predictors.
3. Discount offers and email channel have highest response rates.
4. Over-contacting reduces response — optimize contact frequency.
5. Uplift modeling (who to target) matters more than response modeling.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.9133,
    "f1": 0.9061,
    "precision": 0.9067,
    "recall": 0.9133
  },
  "LightGBM": {
    "accuracy": 0.9158,
    "f1": 0.9099,
    "precision": 0.9098,
    "recall": 0.9158
  },
  "XGBoost": {
    "accuracy": 0.9083,
    "f1": 0.9024,
    "precision": 0.9012,
    "recall": 0.9083
  },
  "Logistic Regression": {
    "accuracy": 0.9058,
    "f1": 0.8982,
    "precision": 0.8977,
    "recall": 0.9058
  },
  "FLAML": {
    "accuracy": 0.9125,
    "f1": 0.9079,
    "precision": 0.9065,
    "recall": 0.9125
  }
}
